# PDCP · N07b — LiteJudge data top-up + Stage-2 balancing (CPU)
**Runs between N08 and N09.** Kaggle: Accelerator **None (CPU)** · Internet **On** · **Save & Run All (Commit)**. Expected ≈ 2–2.5 h. Uses **no GPU quota**.

**Why.** N07 stopped at its time budget after 7000 of 9380 decoupled-pair jobs. Jobs run in family order, so the PDSG pairs have **no WS50 and no TREE50 graphs**, and SBM100 has train graphs only.
This notebook rebuilds N07's job list exactly (same seeds), runs only the **2380 jobs that were never attempted** (Δstress 0.20 skipped: 2 % success), and renders the new drawings.

**Stage 2 (decorrelation).** N07 pairs have r(Δstress, Δcue) ≈ 0.15–0.18 because the conditions thin out unevenly at large Δstress.
The column `in_bal` marks a **balanced subset** (Δstress ≤ 0.10, equal counts per condition within each split × Δstress cell), which gives |r| ≈ 0.01. N09 uses `in_bal` pairs as the PDSG training pool.

**Input:** `kbs-n7-output`. **Outputs** (`/kaggle/working/n07b_output/`): `lj_pairs_full.parquet` (all natural + PDSG pairs, old and new, with `in_bal`), `lj_drawings_new.parquet`, `images/384/*.png` (new drawings only), `n07b_report.json`, `fig_lj_full.png`, `MANIFEST.txt`.

In [1]:
# ==== SETTINGS ====
N07_ROOT = "/kaggle/input/datasets/resadussalehinrafsan/kbs-n07-output"
OUTPUT_DIR = "/kaggle/working/n07b_output"
LJ_SEED = 9_000_000                 # must equal N07
N07_ATTEMPTED = 7000                # N07 log: 'time budget reached' after 7000 jobs
SKIP_DSTRESS = [0.20]               # 2 % success in N07 -> not worth the time
DSTRESS = [0.0, 0.03, 0.06, 0.10, 0.15, 0.20]
CUES = ["nu", "eld", "xr", "mel"]
MARGIN = {"nu": 0.05, "eld": 0.05, "xr": 0.04, "mel": 0.04}
CONDS = {"congruent": (-1, -1, -1, -1), "neutral": (0, 0, 0, 0), "conflict_nu": (1, 0, 0, 0), "conflict_eld": (0, 1, 0, 0),
         "conflict_xr": (0, 0, 1, 0), "conflict_mel": (0, 0, 0, 1), "conflict_all": (1, 1, 1, 1)}
STRESS_FLOOR = 0.295
BAL_MAX_DSTRESS = 0.10              # balanced subset uses Δstress <= this
RES = 384
N_JOBS = 4
MAX_HOURS = 10.0
FAST = False                        # True = 40 jobs only (smoke test)

In [2]:
import os, sys, json, time, glob, shutil, subprocess, importlib
T0 = time.time()
os.makedirs(f"{OUTPUT_DIR}/images/{RES}", exist_ok=True)
for p in ["joblib", "pyarrow"]:
    try: importlib.import_module(p)
    except ImportError: subprocess.run([sys.executable, "-m", "pip", "install", "-q", p], check=True)
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt
from joblib import Parallel, delayed
def hours(): return (time.time() - T0) / 3600
f = sorted(glob.glob(f"{N07_ROOT}/**/lj_pairs.parquet", recursive=True)); assert f, "lj_pairs.parquet not found under N07_ROOT"
N07 = os.path.dirname(f[0]); print("N07:", N07)
shutil.copy(f"{N07}/pdcp_utils.py", f"{OUTPUT_DIR}/pdcp_utils.py")
sys.path.insert(0, OUTPUT_DIR); import pdcp_utils as U; print("utils", U.__version__)
DRW0 = pd.read_parquet(f"{N07}/lj_drawings.parquet"); PAIRS0 = pd.read_parquet(f"{N07}/lj_pairs.parquet")
print(len(DRW0), "drawings |", len(PAIRS0), "pairs (N07)")

N07: /kaggle/input/datasets/resadussalehinrafsan/kbs-n07-output/n07_output
utils 0.1.3
7482 drawings | 15327 pairs (N07)


## 1. Rebuild N07's job list exactly and keep the unattempted jobs

In [3]:
BASE = DRW0[DRW0.role.isin(["base1", "base2"])].reset_index(drop=True)   # same rows and order as N07's BASE
rng = np.random.default_rng(LJ_SEED)
jobs = []
for (fam, n, gi), g in BASE[BASE.role == "base1"].groupby(["family", "n", "graph"]):
    for d in DSTRESS:
        cand = g[g.ksm - d >= STRESS_FLOOR]
        if not len(cand): continue
        for cond in CONDS:
            b = cand.iloc[int(rng.integers(len(cand)))]
            jobs.append({"pair_id": f"D_{fam}{n}_g{gi}_d{int(round(d*100)):02d}_{cond}", "family": fam, "n": int(n), "graph": int(gi),
                         "split": b.split, "graph_seed": int(b.graph_seed), "dstress": d, "cond": cond, "B_id": b.drawing_id,
                         **{"B_" + k: float(b[k]) for k in ["ksm"] + CUES}, "seed": int(rng.integers(1e8))})
ok_old = PAIRS0[PAIRS0.kind == "pdsg"].set_index("pair_id").drawing_B
J = {j["pair_id"]: j["B_id"] for j in jobs}
assert len(jobs) == 9380 and all(J[p] == b for p, b in ok_old.items()), "job list does not match N07 - check N07_ROOT"
assert ok_old.index.isin([j["pair_id"] for j in jobs[:N07_ATTEMPTED]]).all()
TODO = [j for j in jobs[N07_ATTEMPTED:] if j["dstress"] not in SKIP_DSTRESS]
if FAST: TODO = TODO[::max(1, len(TODO) // 40)][:40]
print("job list verified |", len(TODO), "jobs to run |", pd.Series([f"{j['family']}{j['n']}" for j in TODO]).value_counts().to_dict())

job list verified | 2079 jobs to run | {'TREE50': 840, 'WS50': 840, 'SBM100': 399}


## 2. Generate the missing decoupled drawings (same generator and settings as N07)

In [4]:
def gen_A(job):
    import sys; sys.path.insert(0, OUTPUT_DIR)
    import numpy as np, torch, pdcp_utils as U; torch.set_num_threads(1)
    gc = U.GraphCache(U.make_graph(job["family"], job["n"], job["graph_seed"]))
    tg = {"ksm": job["B_ksm"] - job["dstress"]}
    for p, c in zip(CUES, CONDS[job["cond"]]): tg[p] = job["B_" + p] + c * MARGIN[p]
    base = {k: job[k] for k in ["pair_id", "family", "n", "graph", "split", "dstress", "cond", "B_id", "graph_seed"]}
    if min(tg[p] for p in CUES) < 0: return {**base, "ok": False}
    r0 = U.pdsg_optimise(gc, {"ksm": tg["ksm"]}, seed=job["seed"], max_steps=1500, max_restarts=1)
    r = U.pdsg_optimise(gc, tg, X0=r0.X, tol={"ksm": 0.004}, max_steps=1200, max_restarts=1, seed=job["seed"] + 1)
    m = U.all_metrics(r.X, gc, with_realism=True)
    return {**base, "ok": bool(r.converged and m["realistic"]),
            **{k: m[k] for k in ["ksm", "nu", "eld", "xr", "mel", "aspect", "nn_max", "nn_min", "dense"]}, "coords": r.X.ravel().round(6).tolist()}
res = []
for s in range(0, len(TODO), 400):
    if hours() > MAX_HOURS * 0.85: print("time budget reached"); break
    res += Parallel(n_jobs=N_JOBS)(delayed(gen_A)(j) for j in TODO[s:s + 400])
    print(f"{len(res)}/{len(TODO)} | ok {np.mean([r['ok'] for r in res]):.2f} | {hours()*60:.0f} min", flush=True)
GEN = pd.DataFrame(res)
print("success by family:", GEN.assign(fs=GEN.family + GEN.n.astype(str)).groupby("fs").ok.mean().round(2).to_dict())
print("success by condition:", GEN.groupby("cond").ok.mean().round(2).to_dict())
print("success by dstress:", GEN.groupby("dstress").ok.mean().round(2).to_dict())

400/2079 | ok 0.68 | 74 min
800/2079 | ok 0.82 | 79 min
1200/2079 | ok 0.87 | 84 min
1600/2079 | ok 0.88 | 93 min
2000/2079 | ok 0.88 | 104 min
2079/2079 | ok 0.88 | 105 min
success by family: {'SBM100': 0.68, 'TREE50': 0.97, 'WS50': 0.88}
success by condition: {'conflict_all': 0.78, 'conflict_eld': 0.92, 'conflict_mel': 0.9, 'conflict_nu': 0.87, 'conflict_xr': 0.87, 'congruent': 0.88, 'neutral': 0.93}
success by dstress: {0.0: 0.96, 0.03: 0.94, 0.06: 0.96, 0.1: 0.87, 0.15: 0.66}


## 3. Render new drawings, assemble full pair table

In [5]:
GOK = GEN[GEN.ok].copy(); GOK["drawing_id"] = "LA_" + GOK.pair_id.str[2:]
MET = ["ksm", "nu", "eld", "xr", "mel", "aspect", "nn_max", "nn_min", "dense"]
DNEW = GOK.assign(role="A")[["drawing_id", "role", "family", "n", "graph", "graph_seed", "split"] + MET + ["coords"]].reset_index(drop=True)
DNEW["png"] = DNEW.drawing_id.map(lambda d: f"images/{RES}/{d}.png")
def render(rec):
    import sys; sys.path.insert(0, OUTPUT_DIR)
    import numpy as np, pdcp_utils as U
    gc = U.GraphCache(U.make_graph(rec["family"], rec["n"], rec["graph_seed"]))
    U.render_png(gc, np.array(rec["coords"]).reshape(-1, 2), f"{OUTPUT_DIR}/{rec['png']}", res=RES)
    return 1
Parallel(n_jobs=N_JOBS)(delayed(render)(r) for r in DNEW.to_dict("records"))
DNEW.to_parquet(f"{OUTPUT_DIR}/lj_drawings_new.parquet", index=False)

DEC = GOK.assign(kind="pdsg", drawing_A=GOK.drawing_id, drawing_B=GOK.B_id, dstress_design=GOK.dstress)[
    ["pair_id", "kind", "cond", "dstress_design", "family", "n", "graph", "split", "drawing_A", "drawing_B"]]
Di = pd.concat([DRW0, DNEW], ignore_index=True).set_index("drawing_id")
for k in ["ksm", "nu", "eld", "xr", "mel"]:
    DEC["A_" + k] = DEC.drawing_A.map(Di[k]).values; DEC["B_" + k] = DEC.drawing_B.map(Di[k]).values
    DEC["d_" + k] = DEC["B_" + k] - DEC["A_" + k]
DEC["lower"] = np.where(DEC.d_ksm.abs() < 0.005, "same", np.where(DEC.d_ksm > 0, "A", "B"))
PAIRS = pd.concat([PAIRS0.assign(src="n07"), DEC.assign(src="n07b")], ignore_index=True)
PAIRS["img_src"] = "n07"                          # where drawing A's PNG lives; drawing B is always a base drawing (N07)
PAIRS.loc[PAIRS.src == "n07b", "img_src"] = "n07b"
print(len(DNEW), "new drawings |", len(DEC), "new pairs |", len(PAIRS), "pairs total")

1825 new drawings | 1825 new pairs | 17152 pairs total


## 4. Stage 2 — balanced, decorrelated PDSG subset (`in_bal`)
Within each split × Δstress cell (Δstress ≤ 0.10), every condition keeps the same number of pairs (the minimum over conditions); pairs are drawn at random but spread across families.

In [6]:
rng = np.random.default_rng(LJ_SEED + 7)
D = PAIRS[(PAIRS.kind == "pdsg") & (PAIRS.dstress_design <= BAL_MAX_DSTRESS + 1e-9)]
keep = []
for (sp, d), g in D.groupby(["split", "dstress_design"]):
    k = g.groupby("cond").size().min()
    for c, gc_ in g.groupby("cond"):
        gc_ = gc_.assign(_r=rng.random(len(gc_))).sort_values("_r")
        gc_ = gc_.assign(_k=gc_.groupby(["family", "n"]).cumcount()).sort_values(["_k", "_r"])   # round-robin over families
        keep += gc_.head(k).pair_id.tolist()
PAIRS["in_bal"] = PAIRS.pair_id.isin(keep)
def corr(df): return {p: round(float(df.d_ksm.corr(df["d_" + p])), 3) for p in CUES}
CORR = {"natural": corr(PAIRS[PAIRS.kind == "natural"]), "pdsg_all": corr(PAIRS[PAIRS.kind == "pdsg"]),
        "pdsg_balanced": corr(PAIRS[PAIRS.in_bal])}
CORR_SPLIT = {sp: corr(PAIRS[PAIRS.in_bal & (PAIRS.split == sp)]) for sp in ["train", "val", "test"]}
print(json.dumps(CORR)); print(json.dumps(CORR_SPLIT))
PAIRS["fs"] = PAIRS.family + PAIRS.n.astype(str)
TAB = pd.crosstab(PAIRS.fs, [PAIRS.kind.where(~PAIRS.in_bal, "pdsg_bal"), PAIRS.split]); print(TAB)
assert max(abs(v) for v in CORR["pdsg_balanced"].values()) <= 0.05, "balanced subset still correlated"
PAIRS.drop(columns="fs").to_parquet(f"{OUTPUT_DIR}/lj_pairs_full.parquet", index=False)

{"natural": {"nu": 0.557, "eld": 0.352, "xr": 0.801, "mel": 0.76}, "pdsg_all": {"nu": 0.13, "eld": 0.131, "xr": 0.128, "mel": 0.145}, "pdsg_balanced": {"nu": -0.003, "eld": 0.006, "xr": 0.003, "mel": 0.005}}
{"train": {"nu": -0.003, "eld": 0.007, "xr": 0.005, "mel": 0.006}, "val": {"nu": -0.006, "eld": 0.003, "xr": -0.006, "mel": 0.005}, "test": {"nu": 0.005, "eld": 0.009, "xr": -0.0, "mel": 0.003}}
kind   natural            pdsg           pdsg_bal          
split     test train  val test train val     test train val
fs                                                         
BA100      118   623  135   16   112  22       74   391  65
BA50       135   810  135   22   125  21       72   408  72
ER100      118   691  135   10    68  10       66   383  69
ER25       135   810  135   15   129  18       76   405  71
ER50       135   810  135   12    94  11       73   405  73
GRID49     135   810  135   21   144  21       71   407  69
SBM100     101   589  101   14    61  10       65   368  

## 5. Figure, report, manifest

In [7]:
fig, ax = plt.subplots(1, 3, figsize=(17, 3.8), gridspec_kw={"width_ratios": [1.1, 1.3, 1]})
M = np.array([[CORR[k][p] for p in CUES] for k in CORR])
ax[0].imshow(M, cmap="RdBu_r", vmin=-1, vmax=1); ax[0].set_xticks(range(4)); ax[0].set_xticklabels(["Δ NU", "Δ ELD", "Δ XR", "Δ MEL"])
ax[0].set_yticks(range(3)); ax[0].set_yticklabels(["natural", "PDSG (all)", "PDSG balanced"])
for i in range(3):
    for j in range(4): ax[0].text(j, i, f"{M[i,j]:+.2f}", ha="center", va="center")
ax[0].set_title("r(Δstress, Δcue)")
b = PAIRS[PAIRS.in_bal].groupby(["dstress_design", "cond"]).size().unstack(fill_value=0)
ax[1].imshow(b.values, cmap="Greens"); ax[1].set_xticks(range(b.shape[1])); ax[1].set_xticklabels(b.columns, rotation=30, fontsize=7)
ax[1].set_yticks(range(b.shape[0])); ax[1].set_yticklabels(b.index)
for i in range(b.shape[0]):
    for j in range(b.shape[1]): ax[1].text(j, i, b.values[i, j], ha="center", va="center", fontsize=7)
ax[1].set_title("Balanced PDSG pairs per Δstress × condition")
fam = PAIRS[PAIRS.kind == "pdsg"].assign(fs=lambda d: d.family + d.n.astype(str), s=lambda d: np.where(d.in_bal, "balanced", "other"))
fam.groupby(["fs", "s"]).size().unstack(fill_value=0).plot.barh(stacked=True, ax=ax[2]); ax[2].set_title("PDSG pairs per graph set"); ax[2].set_ylabel("")
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/fig_lj_full.png", dpi=130); plt.show()
REP = {"utils": U.__version__, "runtime_min": round(hours() * 60, 1), "jobs_run": len(GEN), "new_pairs": int(len(DEC)),
       "success_by_family": GEN.assign(fs=GEN.family + GEN.n.astype(str)).groupby("fs").ok.mean().round(3).to_dict(),
       "success_by_cond": GEN.groupby("cond").ok.mean().round(3).to_dict(),
       "success_by_dstress": {str(k): v for k, v in GEN.groupby("dstress").ok.mean().round(3).to_dict().items()},
       "pairs": {f"{k}|{s}": int(v) for (k, s), v in PAIRS.groupby(["kind", "split"]).size().items()},
       "balanced_pairs": {s: int(v) for s, v in PAIRS[PAIRS.in_bal].groupby("split").size().items()},
       "correlations": CORR, "balanced_corr_by_split": CORR_SPLIT}
json.dump(REP, open(f"{OUTPUT_DIR}/n07b_report.json", "w"), indent=2, default=str)
shutil.rmtree(f"{OUTPUT_DIR}/__pycache__", ignore_errors=True)
lines = [f"N07b top-up · {REP['runtime_min']} min · {len(DNEW)} new drawings · {len(PAIRS)} pairs total · balanced {int(PAIRS.in_bal.sum())}",
         f"corr: {json.dumps(CORR)}", ""]
for f in sorted(os.listdir(OUTPUT_DIR)):
    p = f"{OUTPUT_DIR}/{f}"
    if os.path.isfile(p): lines.append(f"{f:28s} {os.path.getsize(p)/1024:10.1f} KB")
lines.append(f"images/{RES}/: {len(os.listdir(f'{OUTPUT_DIR}/images/{RES}'))} PNG")
open(f"{OUTPUT_DIR}/MANIFEST.txt", "w").write("\n".join(lines)); print("\n".join(lines))

N07b top-up · 106.4 min · 1825 new drawings · 17152 pairs total · balanced 5404
corr: {"natural": {"nu": 0.557, "eld": 0.352, "xr": 0.801, "mel": 0.76}, "pdsg_all": {"nu": 0.13, "eld": 0.131, "xr": 0.128, "mel": 0.145}, "pdsg_balanced": {"nu": -0.003, "eld": 0.006, "xr": 0.003, "mel": 0.005}}

fig_lj_full.png                    74.9 KB
lj_drawings_new.parquet          1996.7 KB
lj_pairs_full.parquet            1468.2 KB
n07b_report.json                    1.4 KB
pdcp_utils.py                      22.4 KB
images/384/: 1825 PNG
